# ETF 动量策略：调仓频率对收益率的影响

这个 notebook 在 `1~6个月动量窗口 + 防守资产/空仓规则` 的基础上，进一步比较不同调仓频率对收益、回撤、波动和换仓次数的影响。

核心问题：

> 更频繁调仓到底提升了收益，还是只是增加交易成本和噪音？

注意：这是学习和研究样例，不构成投资建议。

## 1. 对比设计

动量窗口：

| 窗口 | 交易日近似 |
|---:|---:|
| 1个月 | 21 |
| 2个月 | 42 |
| 3个月 | 63 |
| 4个月 | 84 |
| 5个月 | 105 |
| 6个月 | 126 |

调仓频率：

| 频率 | 含义 |
|---|---|
| `weekly` | 每周最后一个交易日生成信号 |
| `biweekly` | 每两周最后一个交易日生成信号 |
| `monthly` | 每月最后一个交易日生成信号 |
| `quarterly` | 每季度最后一个交易日生成信号 |

其它规则保持一致：

1. 调仓日收盘后计算信号，下一个交易日生效，避免未来函数；
2. 行业/主题 ETF 中选 N 月动量最高者；
3. 若最佳行业 ETF 动量为正，持有它；
4. 若最佳行业 ETF 动量为负或不可用，转向防守资产；
5. 若最佳防守资产动量也不为正，则空仓；
6. 每次换仓扣 `0.02%` 简化成本。

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

BASE = Path.cwd()
if BASE.name == "notebooks":
    BASE = BASE.parent
DATA_PATH = BASE / "data" / "etf_momentum_daily_eastmoney_qfq.csv"
OUTPUT_DIR = BASE / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

WINDOWS = {
    "mom_1m": 21,
    "mom_2m": 42,
    "mom_3m": 63,
    "mom_4m": 84,
    "mom_5m": 105,
    "mom_6m": 126,
}
REBALANCE_FREQUENCIES = ["weekly", "biweekly", "monthly", "quarterly"]
FEE_RATE = 0.0002

## 2. 读取数据

In [2]:
df = pd.read_csv(DATA_PATH, dtype={"symbol": "string"}, parse_dates=["date"])
df["symbol"] = df["symbol"].astype("string").str.strip()
df["close"] = pd.to_numeric(df["close"], errors="coerce")
df = df.sort_values(["symbol", "date"])

meta = (
    df.groupby("symbol")
    .agg(
        name=("name", "last"),
        bucket=("bucket", "last"),
        theme=("theme", "last"),
        first_date=("date", "min"),
        last_date=("date", "max"),
        rows=("date", "size"),
    )
    .reset_index()
)
meta_by_symbol = meta.set_index("symbol")
close = df.pivot(index="date", columns="symbol", values="close").sort_index()
daily_ret = close.pct_change(fill_method=None).fillna(0.0)

sector_symbols = meta.loc[meta["bucket"].eq("sector"), "symbol"].tolist()
defensive_symbols = meta.loc[meta["bucket"].eq("defensive"), "symbol"].tolist()
benchmark_symbols = meta.loc[meta["bucket"].eq("benchmark"), "symbol"].tolist()

print(f"数据区间: {close.index.min().date()} ~ {close.index.max().date()}")
print(f"ETF 数量: {close.shape[1]}")
print(f"行业/主题 ETF: {len(sector_symbols)}")
print(f"防守资产: {len(defensive_symbols)}", defensive_symbols)
print(f"基准资产: {len(benchmark_symbols)}", benchmark_symbols)

数据区间: 2015-01-05 ~ 2026-06-18
ETF 数量: 18
行业/主题 ETF: 14
防守资产: 2 ['511010', '511260']
基准资产: 2 ['159915', '510300']


## 3. 调仓日与回测函数

In [3]:
def rebalance_dates_for_frequency(index: pd.DatetimeIndex, frequency: str) -> pd.DatetimeIndex:
    dates = pd.Series(index, index=index)
    if frequency == "weekly":
        return pd.DatetimeIndex(dates.groupby(index.to_period("W-FRI")).last().values)
    if frequency == "biweekly":
        weekly = pd.DatetimeIndex(dates.groupby(index.to_period("W-FRI")).last().values)
        return weekly[1::2]
    if frequency == "monthly":
        return pd.DatetimeIndex(dates.groupby(index.to_period("M")).last().values)
    if frequency == "quarterly":
        return pd.DatetimeIndex(dates.groupby(index.to_period("Q")).last().values)
    raise ValueError(f"Unknown frequency: {frequency}")


def max_drawdown(nav: pd.Series) -> float:
    peak = nav.cummax()
    return float((nav / peak - 1.0).min())


def annualized_return(nav: pd.Series, periods_per_year: int = 252) -> float:
    if len(nav) < 2:
        return np.nan
    total = nav.iloc[-1] / nav.iloc[0] - 1.0
    years = len(nav) / periods_per_year
    return float((1.0 + total) ** (1.0 / years) - 1.0)


def annualized_volatility(ret: pd.Series, periods_per_year: int = 252) -> float:
    return float(ret.std() * np.sqrt(periods_per_year))


def sharpe_like(ret: pd.Series, periods_per_year: int = 252) -> float:
    vol = annualized_volatility(ret, periods_per_year)
    if vol == 0 or pd.isna(vol):
        return np.nan
    return float(ret.mean() * periods_per_year / vol)


def run_strategy(label: str, lookback_days: int, frequency: str) -> tuple[pd.DataFrame, pd.DataFrame, dict]:
    signal_dates = rebalance_dates_for_frequency(close.index, frequency)
    momentum = close / close.shift(lookback_days) - 1.0
    decisions = []

    for signal_date in signal_dates:
        row = momentum.loc[signal_date]
        sector_mom = row[sector_symbols].dropna().sort_values(ascending=False)
        defensive_mom = row[defensive_symbols].dropna().sort_values(ascending=False)

        chosen_symbol = "CASH"
        chosen_bucket = "cash"
        chosen_name = "空仓"
        chosen_theme = "现金"
        selected_momentum = 0.0
        reason = "行业和防守资产均无正动量，空仓"
        best_sector_symbol = None
        best_sector_momentum = np.nan
        best_defensive_symbol = None
        best_defensive_momentum = np.nan

        if not sector_mom.empty:
            best_sector_symbol = sector_mom.index[0]
            best_sector_momentum = float(sector_mom.iloc[0])
        if not defensive_mom.empty:
            best_defensive_symbol = defensive_mom.index[0]
            best_defensive_momentum = float(defensive_mom.iloc[0])

        if pd.notna(best_sector_momentum) and best_sector_momentum > 0:
            chosen_symbol = best_sector_symbol
            chosen_bucket = "sector"
            selected_momentum = best_sector_momentum
            reason = "最佳行业/主题 ETF 动量为正，持有该 ETF"
        elif pd.notna(best_defensive_momentum) and best_defensive_momentum > 0:
            chosen_symbol = best_defensive_symbol
            chosen_bucket = "defensive"
            selected_momentum = best_defensive_momentum
            reason = "最佳行业/主题 ETF 动量为负或不可用，切换到正动量防守资产"

        if chosen_symbol != "CASH":
            chosen_name = meta_by_symbol.loc[chosen_symbol, "name"]
            chosen_theme = meta_by_symbol.loc[chosen_symbol, "theme"]

        decisions.append(
            {
                "window_label": label,
                "lookback_days": lookback_days,
                "rebalance_frequency": frequency,
                "signal_date": signal_date,
                "chosen_symbol": chosen_symbol,
                "chosen_name": chosen_name,
                "chosen_bucket": chosen_bucket,
                "chosen_theme": chosen_theme,
                "selected_momentum": selected_momentum,
                "best_sector_symbol": best_sector_symbol,
                "best_sector_momentum": best_sector_momentum,
                "best_defensive_symbol": best_defensive_symbol,
                "best_defensive_momentum": best_defensive_momentum,
                "reason": reason,
            }
        )

    decisions = pd.DataFrame(decisions)
    positions = pd.Series("CASH", index=close.index, name="position", dtype="object")
    decision_by_date = decisions.set_index("signal_date")
    current_position = "CASH"
    for date in close.index:
        positions.loc[date] = current_position
        if date in decision_by_date.index:
            current_position = decision_by_date.loc[date, "chosen_symbol"]

    strategy_ret = pd.Series(0.0, index=close.index, name="strategy_return")
    for symbol in close.columns:
        mask = positions.eq(symbol)
        strategy_ret.loc[mask] = daily_ret.loc[mask, symbol]

    trades = positions.ne(positions.shift(1)).fillna(False)
    trades.iloc[0] = False
    strategy_ret_after_cost = strategy_ret.copy()
    strategy_ret_after_cost.loc[trades] -= FEE_RATE
    nav = (1.0 + strategy_ret_after_cost).cumprod()

    daily = pd.DataFrame(
        {
            "date": close.index,
            "window_label": label,
            "lookback_days": lookback_days,
            "rebalance_frequency": frequency,
            "position": positions.values,
            "strategy_return": strategy_ret.values,
            "strategy_return_after_cost": strategy_ret_after_cost.values,
            "trade": trades.values,
            "nav": nav.values,
        }
    )

    metrics = {
        "window_label": label,
        "lookback_days": lookback_days,
        "rebalance_frequency": frequency,
        "total_return": nav.iloc[-1] / nav.iloc[0] - 1.0,
        "annualized_return": annualized_return(nav),
        "annualized_volatility": annualized_volatility(strategy_ret_after_cost),
        "max_drawdown": max_drawdown(nav),
        "sharpe_like_no_rf": sharpe_like(strategy_ret_after_cost),
        "trade_count": int(trades.sum()),
        "cash_days": int(positions.eq("CASH").sum()),
        "defensive_days": int(positions.isin(defensive_symbols).sum()),
        "sector_days": int(positions.isin(sector_symbols).sum()),
        "final_nav": float(nav.iloc[-1]),
        "signal_count": len(decisions),
    }
    return daily, decisions, metrics

## 4. 执行 24 组对比

In [4]:
all_daily = []
all_decisions = []
all_metrics = []

for label, days in WINDOWS.items():
    for frequency in REBALANCE_FREQUENCIES:
        daily, decisions, metrics = run_strategy(label, days, frequency)
        all_daily.append(daily)
        all_decisions.append(decisions)
        all_metrics.append(metrics)

daily_all = pd.concat(all_daily, ignore_index=True)
decisions_all = pd.concat(all_decisions, ignore_index=True)
metrics = pd.DataFrame(all_metrics).sort_values(["lookback_days", "rebalance_frequency"])

benchmark_symbol = "510300" if "510300" in close.columns else benchmark_symbols[0]
benchmark_nav = (1.0 + daily_ret[benchmark_symbol]).cumprod()
benchmark_metrics = pd.DataFrame(
    [
        {
            "window_label": f"buy_hold_{benchmark_symbol}",
            "lookback_days": 0,
            "rebalance_frequency": "buy_hold",
            "total_return": benchmark_nav.iloc[-1] / benchmark_nav.iloc[0] - 1.0,
            "annualized_return": annualized_return(benchmark_nav),
            "annualized_volatility": annualized_volatility(daily_ret[benchmark_symbol]),
            "max_drawdown": max_drawdown(benchmark_nav),
            "sharpe_like_no_rf": sharpe_like(daily_ret[benchmark_symbol]),
            "trade_count": 0,
            "cash_days": 0,
            "defensive_days": 0,
            "sector_days": len(benchmark_nav),
            "final_nav": float(benchmark_nav.iloc[-1]),
            "signal_count": 0,
        }
    ]
)
metrics_with_benchmark = pd.concat([metrics, benchmark_metrics], ignore_index=True)
metrics_with_benchmark

,window_label,lookback_days,rebalance_frequency,total_return,annualized_return,annualized_volatility,max_drawdown,sharpe_like_no_rf,trade_count,cash_days,defensive_days,sector_days,final_nav,signal_count
0,mom_1m,21,biweekly,1.508387,0.086839,0.372555,-0.645758,0.411521,182,117,145,2521,2.508387,293
1,mom_1m,21,monthly,3.384320,0.143205,0.351948,-0.601498,0.556275,115,155,157,2471,4.384320,138
2,mom_1m,21,quarterly,2.375581,0.116457,0.338361,-0.618754,0.494771,41,180,245,2358,3.375581,46
3,mom_1m,21,weekly,2.577291,0.122339,0.365066,-0.430827,0.498591,260,107,153,2523,3.577291,586
4,mom_2m,42,biweekly,13.603696,0.274799,0.375201,-0.398334,0.835072,118,55,183,2545,14.603696,293
5,mom_2m,42,monthly,11.551456,0.257438,0.379647,-0.526326,0.793678,80,57,120,2606,12.551456,138
6,mom_2m,42,quarterly,2.932890,0.132012,0.369300,-0.618589,0.520951,35,57,61,2665,3.932890,46
7,mom_2m,42,weekly,9.091725,0.232846,0.372977,-0.492277,0.748142,170,70,168,2545,10.091725,586
8,mom_3m,63,biweekly,2.937130,0.132122,0.380387,-0.678682,0.517001,99,64,196,2523,3.937130,293
9,mom_3m,63,monthly,0.291427,0.023428,0.363575,-0.786704,0.246315,71,78,179,2526,1.291427,138


## 5. 对比：同一动量窗口下，不同调仓频率

In [5]:
display_cols = [
    "window_label",
    "rebalance_frequency",
    "annualized_return",
    "max_drawdown",
    "sharpe_like_no_rf",
    "trade_count",
    "signal_count",
    "cash_days",
    "defensive_days",
    "final_nav",
]
metrics[display_cols].sort_values(["window_label", "annualized_return"], ascending=[True, False])

,window_label,rebalance_frequency,annualized_return,max_drawdown,sharpe_like_no_rf,trade_count,signal_count,cash_days,defensive_days,final_nav
2,mom_1m,monthly,0.143205,-0.601498,0.556275,115,138,155,157,4.384320
0,mom_1m,weekly,0.122339,-0.430827,0.498591,260,586,107,153,3.577291
3,mom_1m,quarterly,0.116457,-0.618754,0.494771,41,46,180,245,3.375581
1,mom_1m,biweekly,0.086839,-0.645758,0.411521,182,293,117,145,2.508387
5,mom_2m,biweekly,0.274799,-0.398334,0.835072,118,293,55,183,14.603696
6,mom_2m,monthly,0.257438,-0.526326,0.793678,80,138,57,120,12.551456
4,mom_2m,weekly,0.232846,-0.492277,0.748142,170,586,70,168,10.091725
7,mom_2m,quarterly,0.132012,-0.618589,0.520951,35,46,57,61,3.932890
9,mom_3m,biweekly,0.132122,-0.678682,0.517001,99,293,64,196,3.937130
8,mom_3m,weekly,0.101892,-0.720094,0.445972,144,586,64,198,2.919905


## 6. 对比：全组合排序

In [6]:
metrics[display_cols].sort_values("annualized_return", ascending=False).head(12)

,window_label,rebalance_frequency,annualized_return,max_drawdown,sharpe_like_no_rf,trade_count,signal_count,cash_days,defensive_days,final_nav
5,mom_2m,biweekly,0.274799,-0.398334,0.835072,118,293,55,183,14.603696
6,mom_2m,monthly,0.257438,-0.526326,0.793678,80,138,57,120,12.551456
21,mom_6m,biweekly,0.246696,-0.470170,0.785214,57,293,132,255,11.416856
4,mom_2m,weekly,0.232846,-0.492277,0.748142,170,586,70,168,10.091725
22,mom_6m,monthly,0.224805,-0.479012,0.742470,46,138,142,239,9.388157
17,mom_5m,biweekly,0.211169,-0.567285,0.702602,85,293,113,244,8.296287
18,mom_5m,monthly,0.210550,-0.634847,0.697275,46,138,119,228,8.249558
20,mom_6m,weekly,0.208419,-0.554759,0.699208,89,586,127,261,8.090597
15,mom_4m,quarterly,0.190186,-0.492757,0.660329,27,46,119,180,6.840113
14,mom_4m,monthly,0.168212,-0.710519,0.606326,54,138,98,186,5.567827


In [7]:
metrics[display_cols].sort_values("sharpe_like_no_rf", ascending=False).head(12)

,window_label,rebalance_frequency,annualized_return,max_drawdown,sharpe_like_no_rf,trade_count,signal_count,cash_days,defensive_days,final_nav
5,mom_2m,biweekly,0.274799,-0.398334,0.835072,118,293,55,183,14.603696
6,mom_2m,monthly,0.257438,-0.526326,0.793678,80,138,57,120,12.551456
21,mom_6m,biweekly,0.246696,-0.470170,0.785214,57,293,132,255,11.416856
4,mom_2m,weekly,0.232846,-0.492277,0.748142,170,586,70,168,10.091725
22,mom_6m,monthly,0.224805,-0.479012,0.742470,46,138,142,239,9.388157
17,mom_5m,biweekly,0.211169,-0.567285,0.702602,85,293,113,244,8.296287
20,mom_6m,weekly,0.208419,-0.554759,0.699208,89,586,127,261,8.090597
18,mom_5m,monthly,0.210550,-0.634847,0.697275,46,138,119,228,8.249558
15,mom_4m,quarterly,0.190186,-0.492757,0.660329,27,46,119,180,6.840113
14,mom_4m,monthly,0.168212,-0.710519,0.606326,54,138,98,186,5.567827


## 7. 频率均值：不区分动量窗口

In [8]:
frequency_summary = (
    metrics.groupby("rebalance_frequency")
    .agg(
        avg_annualized_return=("annualized_return", "mean"),
        median_annualized_return=("annualized_return", "median"),
        avg_max_drawdown=("max_drawdown", "mean"),
        avg_sharpe_like=("sharpe_like_no_rf", "mean"),
        avg_trade_count=("trade_count", "mean"),
        best_annualized_return=("annualized_return", "max"),
        worst_annualized_return=("annualized_return", "min"),
    )
    .sort_values("avg_annualized_return", ascending=False)
)
frequency_summary

,avg_annualized_return,median_annualized_return,avg_max_drawdown,avg_sharpe_like,avg_trade_count,best_annualized_return,worst_annualized_return
rebalance_frequency,,,,,,,
biweekly,0.175308,0.171646,-0.564726,0.615714,104.166667,0.274799,0.086839
monthly,0.171273,0.189381,-0.623151,0.607056,68.666667,0.257438,0.023428
weekly,0.160561,0.148935,-0.577838,0.584784,151.666667,0.232846,0.101892
quarterly,0.122516,0.124234,-0.584891,0.498148,32.500000,0.190186,0.035733


## 8. 保存结果

In [9]:
daily_path = OUTPUT_DIR / "etf_momentum_rebalance_frequency_comparison_daily.csv"
decisions_path = OUTPUT_DIR / "etf_momentum_rebalance_frequency_comparison_decisions.csv"
metrics_path = OUTPUT_DIR / "etf_momentum_rebalance_frequency_comparison_metrics.csv"
frequency_summary_path = OUTPUT_DIR / "etf_momentum_rebalance_frequency_summary.csv"

daily_all.to_csv(daily_path, index=False, encoding="utf-8-sig")
decisions_all.to_csv(decisions_path, index=False, encoding="utf-8-sig")
metrics_with_benchmark.to_csv(metrics_path, index=False, encoding="utf-8-sig")
frequency_summary.to_csv(frequency_summary_path, encoding="utf-8-sig")

print(f"已保存每日净值: {daily_path}")
print(f"已保存调仓决策: {decisions_path}")
print(f"已保存绩效指标: {metrics_path}")
print(f"已保存频率汇总: {frequency_summary_path}")

已保存每日净值: d:\Quant\outputs\etf_momentum_rebalance_frequency_comparison_daily.csv
已保存调仓决策: d:\Quant\outputs\etf_momentum_rebalance_frequency_comparison_decisions.csv
已保存绩效指标: d:\Quant\outputs\etf_momentum_rebalance_frequency_comparison_metrics.csv
已保存频率汇总: d:\Quant\outputs\etf_momentum_rebalance_frequency_summary.csv


## 9. 解读提醒

调仓频率不是越高越好：

- 周调仓可能更快识别趋势变化，但也更容易吃到短期噪音；
- 季调仓交易少，但可能错过拐点；
- 月调仓通常是一个折中点；
- 双周调仓可以作为周/月之间的中间方案。

重点不要只看年化收益，也要看换仓次数、最大回撤和防守/空仓天数。  
如果某个频率收益更高但换仓次数暴增，实盘里可能被滑点、冲击成本和最小佣金吞掉。